# LLM-jp-4 8B ThinkingをGoogle Colabで動かす

`llm-jp/llm-jp-4-8b-thinking` を、Google Colab上でbitsandbytesによる4bit量子化を用いて実行します。

Instruct版からの主な変更点:

- `MODEL_ID` を `llm-jp/llm-jp-4-8b-thinking` に変更
- chat templateに `reasoning_effort` を指定
- 推論モデルなので `max_new_tokens` を大きめに設定
- Harmony形式をparserで処理し、通常のチャット画面には最終回答のみを表示
- Gradioから `low / medium / high` のreasoning effortを選択可能

公式モデル:
https://huggingface.co/llm-jp/llm-jp-4-8b-thinking


In [1]:
# =========================================
# 実行環境の準備
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

%pip -q install -U "transformers==5.2.0" "accelerate>=1.13.0" bitsandbytes sentencepiece

import torch, transformers, accelerate
import bitsandbytes as bnb

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("BF16 supported:", torch.cuda.is_bf16_supported())


GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-4fd1ac6b-f253-4aab-6575-318925f062a7)
Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 171.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 84.9 MB/s eta 0:00:00
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
PyTorch: 2.11.0+cu128
CUDA: 12.8
Transformers: 5.2.0
Accelerate: 1.14.0
bitsandbytes: 0.50.0
BF16 supported: True


In [2]:
# =========================================
# Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
CACHE_DIR = PROJECT_DIR / "Program" / "hf_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)
print("CACHE_DIR:", CACHE_DIR)


Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Program/hf_cache


In [3]:
# =========================================
# LLM-jp-4 8B Thinkingモデル
# =========================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "llm-jp/llm-jp-4-8b-thinking"

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR),
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)
model.eval()

INPUT_DEVICE = model.get_input_embeddings().weight.device

print("model loaded:", MODEL_ID)
print("compute dtype:", COMPUTE_DTYPE)
print(f"model memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/63.8k [00:00<?, ?B/s]

llmjp4_tokenizer.py:   0%|          | 0.00/3.88k [00:00<?, ?B/s]

llmjp4_harmony.py:   0%|          | 0.00/4.18k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/llm-jp/llm-jp-4-8b-thinking:
- llmjp4_harmony.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/llm-jp/llm-jp-4-8b-thinking:
- llmjp4_tokenizer.py
- llmjp4_harmony.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

model loaded: llm-jp/llm-jp-4-8b-thinking
compute dtype: torch.bfloat16
model memory footprint: 6.25 GB


In [4]:
# =========================================
# Thinkingモデル用の応答生成関数
# =========================================
@torch.inference_mode()
def chat_generate(
    messages,
    max_new_tokens=1024,
    reasoning_effort="medium",
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        reasoning_effort=reasoning_effort,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(INPUT_DEVICE)

    generation_kwargs = {
        **inputs,
        "max_new_tokens": int(max_new_tokens),
        "do_sample": bool(do_sample),
        "use_cache": True,
    }

    if do_sample:
        generation_kwargs["temperature"] = float(temperature)
        generation_kwargs["top_p"] = float(top_p)

    outputs = model.generate(**generation_kwargs)
    generated_ids = outputs[0, inputs["input_ids"].shape[-1]:].tolist()
    response = tokenizer.decode(generated_ids)

    try:
        parsed = tokenizer.parse_response(response)
        content = parsed.get("content")
        if content:
            return content.strip()
    except Exception as e:
        print("WARNING: parse_response()に失敗しました:", e)

    return response.strip()


In [5]:
# =========================================
# 動作確認
# =========================================
messages = [
    {
        "role": "system",
        "content": "あなたは日本語で簡潔に答える親切なアシスタントです。",
    },
    {
        "role": "user",
        "content": "17×23を計算し、最後に答えだけを簡潔に示してください。",
    },
]

print(chat_generate(
    messages,
    max_new_tokens=1024,
    reasoning_effort="medium",
    do_sample=True,
))


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


391


In [7]:
# =========================================
# Gradioを用いたThinkingモデルのチャットUI
# =========================================
import gradio as gr
import inspect

def make_messages_chatbot(**kwargs):
    if "type" in inspect.signature(gr.Chatbot).parameters:
        kwargs["type"] = "messages"
    return gr.Chatbot(**kwargs)

def gr_chat(history, user_msg):
    history = history or []
    user_msg = (user_msg or "").strip()

    if not user_msg:
        return history, "", history

    messages = history[-8:] + [
        {
            "role": "user",
            "content": user_msg,
        }
    ]

    reply = chat_generate(
        messages,
        max_new_tokens=1024,
        reasoning_effort="medium",
        do_sample=True,
    )

    new_history = history + [
        {
            "role": "user",
            "content": user_msg,
        },
        {
            "role": "assistant",
            "content": reply,
        },
    ]

    return new_history, "", new_history


with gr.Blocks(title="LLM-jp-4 8B Thinking Chat") as chat_demo:
    gr.Markdown(
        "## LLM-jp-4 8B Thinking Chat\n"
    )

    chatbot = make_messages_chatbot(
        label="Chat",
        show_label=False,
        sanitize_html=True,
    )

    chat_state = gr.State([])

    user_box = gr.Textbox(
        placeholder="質問を入力してください。",
        label="",
    )

    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear")

    send_btn.click(
        gr_chat,
        inputs=[chat_state, user_box],
        outputs=[chat_state, user_box, chatbot],
        queue=False,
    )

    clear_btn.click(
        lambda: ([], "", []),
        outputs=[chat_state, user_box, chatbot],
    )

print("WARNING: share=Trueで公開URLが作成されます。")

chat_demo.launch(
    share=True,
    inline=True,
    debug=False,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://32c17bd3dd94ef1a7e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
